# AgroVision MobileNetV2 Foliar Pathology Model Training Pipeline
**Author / Model Owner:** UGDS Munasinghe (SE205.3 Machine Learning Pipeline)  
**Review Partner:** MASN Weerasinghe  

This notebook trains an on-device MobileNetV2 foliar disease classifier for smallholder farmers in Sri Lanka, covering the 18 canonical classes across Tomato, Potato, Pepper, Chilli, Eggplant, and foliar nutrient deficiencies.

In [ ]:
# 1. Environment & Hardware Verification
import os
import json
import time
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")

RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# 2. Dataset Setup and Class Configuration
DATA_DIR = Path("data/processed")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_dir = DATA_DIR / "train"
val_dir = DATA_DIR / "val"

# Canonical 18 classes
CANONICAL_CLASSES = [
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Bacterial_spot",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites_Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Pepper___Bacterial_spot",
    "Pepper___healthy",
    "Chilli___Leaf_curl",
    "Eggplant___healthy",
    "General___Nitrogen_deficiency"
]

print(f"Target Classes: {len(CANONICAL_CLASSES)}")

In [ ]:
# 3. Data Augmentation Pipeline for Outdoor Resilience
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
], name="data_augmentation")

In [ ]:
# 4. Build MobileNetV2 Architecture
def build_model(num_classes=18):
    inputs = layers.Input(shape=(224, 224, 3), name="input_image")
    x = layers.Rescaling(scale=1./127.5, offset=-1.0)(inputs)
    x = data_augmentation(x)
    
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
        alpha=1.0
    )
    base_model.trainable = False
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(0.2, name="dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="AgroVision_MobileNetV2")
    return model, base_model

model, base_model = build_model(len(CANONICAL_CLASSES))
model.summary()

In [ ]:
# 5. Phase 1: Train Classification Head
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top_3_accuracy")]
)

# Callbacks
early_stopping = callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)
checkpoint = callbacks.ModelCheckpoint("ml/models/best_model.keras", monitor="val_accuracy", save_best_only=True)

print("Ready for Phase 1 feature extraction training.")

In [ ]:
# 6. Phase 2: Fine-Tuning Top Convolutional Layers
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top_3_accuracy")]
)
print("Top layers unfrozen for fine-tuning at lr=1e-5.")

In [ ]:
# 7. Export to Quantized TensorFlow Lite (.tflite)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

out_path = Path("app/assets/model/model.tflite")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "wb") as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f"Quantized TFLite model saved to {out_path} ({size_mb:.2f} MB)")